# 06 — Otimização de Hiperparâmetros via Algoritmo Genético

**Tech Challenge Fase 2 — Projeto 1**

Este notebook continua o trabalho da Fase 1. A ideia aqui é pegar o modelo tabular já usado no projeto e testar uma busca de hiperparâmetros com **Algoritmo Genético**, em vez de depender só de escolhas manuais ou GridSearch.

O foco fica no **XGBoost**, porque ele já era um dos modelos principais da Fase 1 e tem bastante parâmetro que pode ser ajustado pelo AG.

O fluxo geral é:

1. carregar e pré-processar os dados usando o pipeline já existente;
2. separar uma amostra estratificada para deixar a busca do AG viável;
3. rodar três configurações diferentes de algoritmo genético;
4. comparar os modelos otimizados com o modelo original;
5. salvar o melhor XGBoost otimizado para ser usado no notebook da LLM.


In [1]:
import sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")

# Permite rodar o notebook tanto pela pasta raiz quanto de dentro de notebooks/.
# Isso evita ter que mudar import manualmente dependendo de onde o Jupyter foi aberto.
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.tabular.load_data import carregar_dataset
from src.tabular.preprocessing import executar_pipeline_preprocessamento
from src.genetic.fitness import FitnessEvaluator
from src.genetic.ga_optimizer import GeneticOptimizer, GAConfig
from src.genetic.chromosome import decode
from src.monitoring import ExperimentTracker

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Projeto:", ROOT)


Projeto: /Users/arthuraugustopaulahardman/projetos/ml-tech


## 1. Carregar dados e pré-processar

Nesta parte reutilizamos o pipeline da Fase 1. Assim o AG trabalha em cima da mesma base tratada que já foi usada para treinar e avaliar os modelos originais.


In [2]:
# Carrega o dataset bruto do SIVEP e aplica o mesmo tratamento da Fase 1.
# O parâmetro salvar=True mantém os artefatos processados para os próximos passos do projeto.
df = carregar_dataset()
artefatos = executar_pipeline_preprocessamento(df, salvar=True)

X_train, y_train = artefatos["X_train"], artefatos["y_train"]
X_test,  y_test  = artefatos["X_test"],  artefatos["y_test"]

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Distribuição treino:", dict(y_train.value_counts()))


Dataset carregado: 267,984 registros, 194 colunas
Registros após filtro de EVOLUCAO válida: 240,436 (89.7% do total)


Distribuição do target binário — Cura: 219,708 (91.4%) | Óbito: 20,728 (8.6%)
Split — Treino: 168,304 | Validação: 36,066 | Teste: 36,066


Valores ausentes após imputação — máx. por coluna (treino): 0.00%


Colunas codificadas: ['CS_SEXO']


Artefatos salvos em: /Users/arthuraugustopaulahardman/projetos/ml-tech/src/data/processed
Treino: (168304, 37) | Teste: (36066, 37)
Distribuição treino: {0: 153794, 1: 14510}


## 2. Amostra estratificada para a busca do AG

Rodar cross-validation com XGBoost em todos os registros, para cada indivíduo e a cada geração, pode ficar bem pesado. Por isso a busca usa uma amostra estratificada do treino.

Depois que o AG encontra bons hiperparâmetros, o modelo final é treinado novamente usando o treino completo.


In [3]:
from sklearn.model_selection import train_test_split

CFG = yaml.safe_load(open(ROOT / "experiments" / "experimentos.yaml"))
defaults = CFG["defaults"]
sample_size = min(defaults["fitness_sample_size"], len(X_train))

# A amostra é estratificada para manter a proporção de cura/óbito parecida com a base original.
# Isso deixa o teste mais rápido sem distorcer demais o problema que o modelo está aprendendo.
if sample_size < len(X_train):
    X_fit, _, y_fit, _ = train_test_split(
        X_train, y_train, train_size=sample_size,
        stratify=y_train, random_state=defaults["random_state"],
    )
else:
    X_fit, y_fit = X_train, y_train

print("Amostra de busca:", X_fit.shape, "| Óbitos:", int(y_fit.sum()))


Amostra de busca: (30000, 37) | Óbitos: 2586


## 3. Executar os 3 experimentos de AG

Aqui ficam os três cenários pedidos na Fase 2. Cada experimento muda configurações como população, gerações, taxa de crossover e taxa de mutação.

A cada geração o tracker salva o melhor resultado, a média da população e o desvio. Isso ajuda a mostrar se o AG realmente está evoluindo ou se ficou parado.


In [4]:
resultados = {}
trackers = {}

for exp in CFG["experiments"]:
    name = exp["name"]
    print(f"\n{'='*70}\n{name} — {exp['descricao']}\n{'='*70}")

    # O evaluator calcula o fitness de cada indivíduo.
    # Neste caso, cada indivíduo é um conjunto de hiperparâmetros do XGBoost.
    evaluator = FitnessEvaluator(
        X_fit, y_fit,
        metric=defaults["metric"],
        cv_folds=defaults["cv_folds"],
        random_state=defaults["random_state"],
    )

    # Configuração do AG para o experimento atual.
    # Esses valores vêm do YAML para facilitar testar cenários diferentes sem mexer no código.
    ga_cfg = GAConfig(
        population_size=exp["population_size"],
        generations=exp["generations"],
        crossover_rate=exp["crossover_rate"],
        mutation_rate=exp["mutation_rate"],
        mutation_sigma=exp["mutation_sigma"],
        tournament_k=exp["tournament_k"],
        elitism_size=exp["elitism_size"],
        n_jobs=defaults["n_jobs"],
        random_state=defaults["random_state"],
        name=name,
    )

    tracker = ExperimentTracker(name)
    ga = GeneticOptimizer(evaluator, ga_cfg, on_generation=tracker.log_generation)

    t0 = time.time()
    res = ga.run(verbose=True)
    elapsed = time.time() - t0

    # Salva o histórico do experimento e o gráfico individual de convergência.
    tracker.save(res)
    tracker.plot_convergence()

    resultados[name] = res
    trackers[name] = tracker

    print(f"\n>>> {name}: melhor F1 (busca) = {res.best_fitness:.4f} em {elapsed:.0f}s")
    print("    Hiperparâmetros:", decode(res.best_individual, random_state=defaults["random_state"]))



exp_A — Baseline leve/rápido


[exp_A] geração 1/8 | melhor=0.4878 | média=0.4537 | desvio=0.0204


[exp_A] geração 2/8 | melhor=0.4947 | média=0.4782 | desvio=0.0177


[exp_A] geração 3/8 | melhor=0.4947 | média=0.4913 | desvio=0.0023


[exp_A] geração 4/8 | melhor=0.4947 | média=0.4904 | desvio=0.0065


...

[exp_C] geração 8/12 | melhor=0.5101 | média=0.4971 | desvio=0.0210


[exp_C] geração 9/12 | melhor=0.5112 | média=0.5027 | desvio=0.0084


[exp_C] geração 10/12 | melhor=0.5112 | média=0.5031 | desvio=0.0154


[exp_C] geração 11/12 | melhor=0.5112 | média=0.5027 | desvio=0.0145


[exp_C] geração 12/12 | melhor=0.5126 | média=0.4988 | desvio=0.0196

>>> exp_C: melhor F1 (busca) = 0.5126 em 74s
    Hiperparâmetros: {'n_estimators': 550, 'max_depth': 3, 'learning_rate': 0.0424, 'subsample': 0.7693, 'colsample_bytree': 1.0, 'min_child_weight': 9, 'gamma': 3.8292, 'scale_pos_weight': 2.6133}


## 4. Curvas de convergência

Esse gráfico compara os três experimentos. O ideal é ver o melhor fitness subindo ao longo das gerações, ou pelo menos estabilizando em um valor melhor do que o começo.


In [5]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))

for name, res in resultados.items():
    gens = [r.generation + 1 for r in res.history]
    best = [r.best_fitness for r in res.history]
    ax.plot(gens, best, marker="o", label=f"{name} (best)")

ax.set_xlabel("Geração")
ax.set_ylabel("Melhor F1 (busca)")
ax.set_title("Convergência do AG — comparativo entre experimentos")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(ROOT / "results" / "figures" / "ga_convergence_comparativo.png", dpi=120)
plt.show()


## 5. Retreinar o melhor de cada experimento

A busca do AG usa uma amostra para ser mais rápida. Agora pegamos o melhor indivíduo de cada experimento, decodificamos os hiperparâmetros e treinamos o XGBoost com o treino completo.

A avaliação final é feita no conjunto de teste, que não foi usado durante a busca.


In [6]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from xgboost import XGBClassifier

def avaliar(model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    return {
        "accuracy":  accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall":    recall_score(y, pred, zero_division=0),
        "f1":        f1_score(y, pred, zero_division=0),
        "roc_auc":   roc_auc_score(y, proba),
    }

linhas = []
modelos_ga = {}

for name, res in resultados.items():
    # Transforma o cromossomo vencedor em parâmetros reais do XGBoost.
    kwargs = decode(res.best_individual, random_state=defaults["random_state"])

    model = XGBClassifier(**{**kwargs, "n_jobs": -1})
    model.fit(X_train, y_train)

    modelos_ga[name] = model

    m = avaliar(model, X_test, y_test)
    m["modelo"] = f"XGBoost-AG ({name})"
    linhas.append(m)

df_ga = pd.DataFrame(linhas).set_index("modelo")
df_ga


,accuracy,precision,recall,f1,roc_auc
modelo,,,,,
XGBoost-AG (exp_A),0.911274,0.486876,0.542940,0.513382,0.905521
XGBoost-AG (exp_B),0.899601,0.440741,0.612416,0.512586,0.905463
XGBoost-AG (exp_C),0.914019,0.501230,0.524284,0.512498,0.905930


## 6. Comparativo com o XGBoost original da Fase 1

Aqui comparamos o XGBoost otimizado pelo AG com o XGBoost original salvo na Fase 1.

Se o arquivo `xgboost.pkl` ainda não existir, basta rodar primeiro o pipeline principal da Fase 1.


In [7]:
import joblib

orig_path = ROOT / "results" / "models" / "xgboost.pkl"
linha_orig = None

if orig_path.exists():
    try:
        orig = joblib.load(orig_path)
        m = avaliar(orig, X_test, y_test)
        m["modelo"] = "XGBoost-original (Fase 1)"
        linha_orig = m
        print("Modelo original da Fase 1 carregado.")
    except Exception as e:
        print("Não foi possível avaliar o modelo original:", e)
else:
    print("xgboost.pkl da Fase 1 não encontrado — rode o pipeline da Fase 1 para comparar.")

# A tabela final deixa claro o ganho ou perda do AG em relação ao modelo original.
tabela = df_ga.copy()
if linha_orig is not None:
    tabela = pd.concat([pd.DataFrame([linha_orig]).set_index("modelo"), tabela])

tabela = tabela[["accuracy", "precision", "recall", "f1", "roc_auc"]].round(4)
tabela


Modelo original da Fase 1 carregado.


,accuracy,precision,recall,f1,roc_auc
modelo,,,,,
XGBoost-original (Fase 1),0.8523,0.3301,0.6928,0.4471,0.8797
XGBoost-AG (exp_A),0.9113,0.4869,0.5429,0.5134,0.9055
XGBoost-AG (exp_B),0.8996,0.4407,0.6124,0.5126,0.9055
XGBoost-AG (exp_C),0.9140,0.5012,0.5243,0.5125,0.9059


In [8]:
# Escolhemos o melhor experimento usando F1 no teste.
# Esse modelo fica salvo para ser usado no notebook 07, onde a LLM gera explicações.
melhor_exp = df_ga["f1"].idxmax().split("(")[-1].strip(")")
print("Melhor experimento por F1 no teste:", melhor_exp)

melhor_modelo = modelos_ga[melhor_exp]

out = ROOT / "results" / "models" / "xgboost_ga_optimized.pkl"
joblib.dump(melhor_modelo, out)
print("Modelo otimizado salvo em:", out)


Melhor experimento por F1 no teste: exp_A
Modelo otimizado salvo em: /Users/arthuraugustopaulahardman/projetos/ml-tech/results/models/xgboost_ga_optimized.pkl


## 7. Conclusões

- Os três experimentos atendem ao requisito de **≥3 configurações de AG**.
- O AG testa diferentes combinações de hiperparâmetros do XGBoost usando população, seleção, crossover, mutação e elitismo.
- A curva de convergência mostra como o melhor fitness evoluiu em cada experimento.
- A tabela final permite comparar o modelo original da Fase 1 com os modelos otimizados.
- O melhor modelo é salvo como `xgboost_ga_optimized.pkl`, que será usado na etapa de explicação com LLM.
